# Lecture 6.2 — Viewing traces in the OpenAI Dashboard

**Section 06 — Tracing, Observability & Capstone**

In the previous lecture you learned that tracing is switched on by default, and you saw the
span hierarchy on a diagram. This notebook is where that diagram turns into something you can
click through.

The notebook itself is deliberately short. Its only job is to **generate rich trace data**. The
real teaching happens in the browser, at the OpenAI Traces dashboard, once these cells have run.

By the end of this notebook you will have produced:

1. **One trace containing every span type** from the hierarchy in Lecture 6.1.
2. **A pair of traces linked by a shared `group_id`**, representing one conversation across two
   separate runs.
3. **A direct dashboard URL for each of them**, printed before the run even finishes.

## Cell 1: Install the Agents SDK

This cell installs the OpenAI Agents SDK into the current runtime. That is the only package this
notebook needs. Everything else it uses is either part of the SDK or part of the standard library.

The version is pinned so that every example in this notebook behaves exactly as recorded. If the
package is already present in this session, pip will simply confirm it and move on, which takes
a couple of seconds.

The `-q` flag keeps the install output quiet so the cell finishes with a clean log.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.6 MB/s eta 0:00:00


## Cell 2: Configure your OpenAI API key

The SDK reads your credentials from the `OPENAI_API_KEY` environment variable. In Google Colab
the safe way to supply that value is the built-in **Secrets** manager, which stores the key
against your Google account rather than inside the notebook file.

**To add your key in Colab:**

| Step | Action |
|---|---|
| 1 | Click the **key icon** in the left sidebar to open the Secrets panel. |
| 2 | Click **Add new secret**. |
| 3 | Set the **Name** to `OPENAI_API_KEY` exactly, including the underscores. |
| 4 | Paste your key into the **Value** field. |
| 5 | Turn on the **Notebook access** toggle for this notebook. |

The cell below reads that secret and copies it into the environment where the SDK will look for
it. Your key never appears in the notebook and never gets committed anywhere.

> **Running locally instead of in Colab?** Skip this cell and set the variable in your terminal
> before launching Jupyter: `export OPENAI_API_KEY="your-key-here"` on macOS or Linux, or
> `setx OPENAI_API_KEY "your-key-here"` on Windows.

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

API key loaded: True


## Cell 3: Choose the model

Every agent in this notebook reads its model from a single variable. Change the value here once
and the whole notebook follows, including all four agents defined later.

`gpt-5.4-mini` is fast and inexpensive, which matters in this lecture because you are about to
make several runs purely to generate trace data.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

print("Using model:", MODEL_NAME)

Using model: gpt-5.4-mini


## Cell 4: Imports

Everything this notebook needs, gathered in one place.

| Import | What it is for |
|---|---|
| `BaseModel` | Structured output type for the guardrail's classifier agent. |
| `Reasoning` | Reasoning effort setting. Note it comes from `openai.types.shared`, **not** from `agents`. |
| `Agent` | The agent class. |
| `GuardrailFunctionOutput` | The value every guardrail function must return. |
| `ModelSettings` | Wraps reasoning effort and verbosity. |
| `RunConfig` | Carries the tracing settings: `workflow_name`, `trace_id`, `group_id`, `trace_metadata`. |
| `RunContextWrapper` | First argument passed to every guardrail function. |
| `Runner` | Executes an agent run. |
| `TResponseInputItem` | Type of the items in a conversation input list. |
| `function_tool` | Turns a plain Python function into a tool the model can call. |
| `gen_trace_id` | **New in this lecture.** Generates a correctly formatted trace ID. |
| `input_guardrail` | Decorator that turns a function into an input guardrail. |
| `output_guardrail` | Decorator that turns a function into an output guardrail. |

Only `gen_trace_id` is new. Everything else you have already used across Sections 3 and 5.

In [4]:
from pydantic import BaseModel
from typing import Literal
from openai.types.shared import Reasoning
from agents import (
    Agent,
    GuardrailFunctionOutput,
    ModelSettings,
    RunConfig,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    function_tool,
    gen_trace_id,
    input_guardrail,
    output_guardrail,
)

print("Imports ready.")

Imports ready.


## Cell 5: What we are building

This cell has no code. It explains the plan before you run anything else.

In Lecture 6.1 you saw the default span hierarchy on a diagram: a trace wraps a `task_span`, which
wraps `turn_span`, which wraps `agent_span`, and nested inside those sit `generation_span`,
`function_span`, `guardrail_span`, and `handoff_span`.

A trivial one-agent run only produces a few of those. To see the whole hierarchy in the dashboard,
you need a system that exercises every branch of it. So the next cell builds one deliberately:

| Component | Span type it produces |
|---|---|
| Two agents, one handing off to the other | `agent_span` (one per agent) |
| Every model call made by either agent | `generation_span` |
| A `@function_tool` the specialist must call | `function_span` |
| An `@input_guardrail` on the triage agent | `guardrail_span` |
| An `@output_guardrail` on the specialist | `guardrail_span` |
| `handoffs=[...]` on the triage agent | `handoff_span` |

**Nothing in that build is new.** The input guardrail follows the pattern from Lecture 5.8, the
output guardrail follows Lecture 5.9, and the handoff follows Lecture 5.2. It is assembled
entirely from parts you already know.

The point of this lecture is not the code. It is the trace the code produces.

## Cell 6: Build the multi-agent system

This is a recap build. Read it quickly and move on. Every pattern here has appeared before.

The system is a small store support desk:

- **`front_door`** is the triage agent. It carries the input guardrail and hands off policy
  questions to the specialist.
- **`policy_specialist`** answers the question. It carries the function tool and the output
  guardrail.
- **`topic_agent`** is the small classifier the input guardrail runs internally, exactly as in
  Lecture 5.8.

Two details worth noticing as you read:

1. Every agent is given `model=MODEL_NAME`. No model string is written inline anywhere.
2. Every agent gets the same `ModelSettings`. Reasoning effort is set to `"none"` and verbosity
   to `"low"` so the runs stay fast and cheap. You are paying for trace data here, not for
   elaborate answers.

In [5]:
class TopicCheck(BaseModel):
    is_on_topic: bool
    reason: str


topic_agent = Agent(
    name="Topic classifier",
    instructions=(
        "Decide whether the user message concerns an online store. "
        "Orders, shipping, delivery, returns, refunds and products are on topic. "
        "Everything else is off topic."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=TopicCheck,
)


@input_guardrail
async def on_topic_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    user_input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:
    result = await Runner.run(topic_agent, user_input, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=not result.final_output.is_on_topic,
    )


@output_guardrail
async def length_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    agent_output: str,
) -> GuardrailFunctionOutput:
    too_long = len(agent_output) > 1200
    return GuardrailFunctionOutput(
        output_info={"characters": len(agent_output), "too_long": too_long},
        tripwire_triggered=too_long,
    )


@function_tool
def lookup_policy(topic: Literal["refunds", "shipping", "delivery"]) -> str:
    """Look up the published store policy for a topic.

    Args:
        topic: The policy area to look up.
    """
    policies = {
        "refunds": "Damaged items are refunded in full within 30 days of delivery.",
        "shipping": "We ship to 40 countries outside the United States.",
        "delivery": "International orders arrive in 7 to 12 business days.",
    }
    return policies.get(topic.lower(), "No published policy found for that topic.")

policy_specialist = Agent(
    name="Policy specialist",
    instructions=(
        "You answer store policy questions. "
        "Always call the lookup_policy tool before you answer. "
        "Keep your answer under four sentences."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[lookup_policy],
    output_guardrails=[length_guardrail],
)


front_door = Agent(
    name="Front door",
    instructions=(
        "You are the first point of contact for an online store. "
        "Hand off to the Policy specialist only for questions about "
        "refunds, shipping, or delivery policy. "
        "For anything else, such as product availability or stock questions, "
        "answer directly yourself and do not hand off."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    handoffs=[policy_specialist],
    input_guardrails=[on_topic_guardrail],
)

print("Triage agent:    ", front_door.name)
print("Hands off to:    ", policy_specialist.name)
print("Input guardrail: ", front_door.input_guardrails[0].get_name())
print("Output guardrail:", policy_specialist.output_guardrails[0].get_name())
print("Specialist tool: ", policy_specialist.tools[0].name)

Triage agent:     Front door
Hands off to:     Policy specialist
Input guardrail:  on_topic_guardrail
Output guardrail: length_guardrail
Specialist tool:  lookup_policy


## Cell 7: Generate the trace ID before the run

Normally the SDK generates a trace ID for you when the run starts, and you go looking for your
run in the dashboard afterwards. There is a better way.

`gen_trace_id()` produces a correctly formatted ID on demand. The format is `trace_` followed by
32 hexadecimal characters, which is exactly what the dashboard expects. Because you generate it
yourself, you know the ID before the run exists.

That means you can build the dashboard URL up front. The pattern is:

```
https://platform.openai.com/traces/trace?trace_id={trace_id}
```

Run this cell and the link will be sitting in your output, ready, before you have run a single
agent.

In [6]:
trace_id = gen_trace_id()

print("Trace ID:", trace_id)
print()
print("View trace:")
print(f"https://platform.openai.com/traces/trace?trace_id={trace_id}")

Trace ID: trace_5a3b9260603c4e76bda820ab17cb06ae

View trace:
https://platform.openai.com/traces/trace?trace_id=trace_5a3b9260603c4e76bda820ab17cb06ae


## Cell 8: Run the system under that trace ID

Now hand the ID you just generated to `RunConfig`, so the run reports itself under that exact
trace instead of inventing a new one.

Three `RunConfig` fields are doing the work here, all of them first covered in Lecture 4.4:

| Field | Effect in the dashboard |
|---|---|
| `workflow_name` | The label the trace appears under. This is why Lecture 4.4 insisted on always setting it. Left unset, every trace is called "Agent workflow". |
| `trace_id` | Forces the run to use your ID, which is what makes the printed link resolve. |
| `trace_metadata` | Arbitrary key-value pairs attached to the trace root, visible when you open it. Useful for tagging runs by environment, user, or experiment. |

The question is deliberately a policy question, so the run exercises the whole system: the input
guardrail classifies it, `front_door` hands off, `policy_specialist` calls its tool, and the
output guardrail checks the answer.

**One thing to expect.** The default `BatchTraceProcessor` exports traces in the background every
few seconds, so the trace may take a moment to appear after the cell finishes. If the link opens
to nothing, wait a few seconds and reload.

In [9]:
result = await Runner.run(
    front_door,
    "What is your full refund policy for damaged items?",
    #"Do you have the wireless keyboard in stock right now?",
    #"Whats the weather like?",
    run_config=RunConfig(
        workflow_name="Dashboard walkthrough",
        trace_id=trace_id,
        trace_metadata={
            "lecture": "6.2",
            "demo": "span-tour",
        },
    ),
)

print("Final output:", result.final_output[:150])
print("Last agent:  ", result.last_agent.name)
print()
print("Open this trace:")
print(f"https://platform.openai.com/traces/trace?trace_id={trace_id}")

Final output: Damaged items are eligible for a full refund if requested within 30 days of delivery.
Last agent:   Policy specialist

Open this trace:
https://platform.openai.com/traces/trace?trace_id=trace_5a3b9260603c4e76bda820ab17cb06ae


## Cell 9: Two runs, one conversation, one group_id

You have been setting `group_id` since Lecture 5.6 without ever seeing what it does. This cell is
where that pays off.

Each call to `Runner.run()` creates its own separate trace. That is a problem when a single
conversation spans several runs, because the dashboard would show them as unrelated events with
no indication they belong together.

`group_id` solves it. Any traces sharing a `group_id` are linked in the dashboard as one thread.

This cell makes two runs. The second one continues the first by passing
`result_a.to_input_list()` plus a follow-up question, which is the multi-turn pattern from
Section 4. Both runs get their own generated `trace_id`, and both get the **same** `group_id`.

In [8]:
session_group = "dashboard-demo-session-001"

trace_id_a = gen_trace_id()
trace_id_b = gen_trace_id()

result_a = await Runner.run(
    front_door,
    "Do you ship internationally?",
    run_config=RunConfig(
        workflow_name="Dashboard walkthrough",
        trace_id=trace_id_a,
        group_id=session_group,
    ),
)

result_b = await Runner.run(
    front_door,
    result_a.to_input_list() + [
        {"role": "user", "content": "And how long does it take?"}
    ],
    run_config=RunConfig(
        workflow_name="Dashboard walkthrough",
        trace_id=trace_id_b,
        group_id=session_group,
    ),
)

print("Group ID:", session_group)
print()
print("Turn 1 trace:", trace_id_a)
print(f"https://platform.openai.com/traces/trace?trace_id={trace_id_a}")
print()
print("Turn 2 trace:", trace_id_b)
print(f"https://platform.openai.com/traces/trace?trace_id={trace_id_b}")
print()
print("Two separate runs, two separate traces, one group_id.")
print("Turn 2 answer:", result_b.final_output[:150])

Group ID: dashboard-demo-session-001

Turn 1 trace: trace_0efa5bbdd65245c788d13dd85822c954
https://platform.openai.com/traces/trace?trace_id=trace_0efa5bbdd65245c788d13dd85822c954

Turn 2 trace: trace_ab94b5220bcd43938ca277da53e9d05b
https://platform.openai.com/traces/trace?trace_id=trace_ab94b5220bcd43938ca277da53e9d05b

Two separate runs, two separate traces, one group_id.
Turn 2 answer: International orders arrive in 7 to 12 business days.


## Cell 10: Reading guide

This cell has no code. Keep it open in a second tab while you work through the dashboard.

Each span type answers a different debugging question. Here is what each one carries and what it
tells you when something has gone wrong.

| Span | What it contains | What it tells you |
|---|---|---|
| **Trace root** | `workflow_name`, `group_id`, `metadata`, `trace_id` | Which workflow this run was, and what it belongs with |
| **`agent_span`** | Which agent ran, plus start and end time | The routing path taken through your system |
| **`generation_span`** | The LLM input and output | Exactly what the model saw and exactly what it said back |
| **`function_span`** | The tool input and output | Which arguments the model chose to pass |
| **`guardrail_span`** | `name`, `triggered` | Whether a tripwire fired, and which guardrail it was |
| **`handoff_span`** | Source agent and target agent | Where control transferred, and to whom |
| **`task_span` / `turn_span`** | Runner invocation and model turn boundaries | The structural nesting of the run |

**Two caveats worth knowing before you go looking.**

**Sensitive data.** `generation_span` and `function_span` are the spans that store real payloads,
so they are the ones covered by `RunConfig.trace_include_sensitive_data`. It defaults to `True`.
If you set it to `False`, or set the `OPENAI_AGENTS_TRACE_INCLUDE_SENSITIVE_DATA` environment
variable, those spans still appear in the dashboard but their contents are empty. That is not a
broken dashboard. That is the setting working as designed.

**Structural spans.** If the `task_span` and `turn_span` layers add more nesting than you want,
they can be switched off per run with
`RunConfig(tracing={"include_task_and_turn_spans": False})`. The agent, generation, tool and
guardrail spans are unaffected.

---

## Where this leaves you

You now have three traces sitting in the dashboard, and the vocabulary to read any of them.

The dashboard is the fastest debugging tool in the SDK, and it required no setup at all. Tracing
was already on. All this notebook did was give it something interesting to record.

**Next up, Lecture 6.3:** the `trace()` context manager, which lets you wrap several runs inside
one single trace instead of grouping separate ones after the fact.